<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Module 5: *Subset and Split*
##### Version Number: 4.0
---
### Contents
> *One Hot Encoding*\
> *Separate Dates for modeling*\
> *Downsize Data*\
> *Split Data*\
> *Export Data*
---
### Notes
---
### Inputs
- engineered_samples.csv - Complete main dataset
---
### Outputs 
- `X_ignition`,`X_damage`, `X_spread`, `pal_X` - numeric and bool columns for modeling
- `y_ignition`,`y_spread`,`y_damage` - target data
- `ignition_details`,`spread_details`,`damage_details`,`pal_details` - text columns and spatial info
---
### User Defined Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [2]:
# Core data tools
import pandas as pd
import numpy as np
from datetime import datetime

from rich.console import Console
from rich.table import Table
from rich.columns import Columns

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

---

###  Load Data

In [3]:
samples = pd.read_csv("../data/processed/engineered_samples.csv")

In [4]:
console = Console()

In [5]:
samples['State Region'] = 'Southern'
samples.loc[samples['centroid_northing'] > -113000,'State Region'] = 'Northern'

In [6]:
samples = samples.sort_values(
    by=["grid_id", "Date"]
)

## One Hot Encoding

One hot encode appropriate categorical variables.

In [7]:
categorical_columns = ['dominant_province_description','dominant_section_description','Season', 'State Region']
one_hot = samples[categorical_columns]
samples = samples.drop(columns = ['dominant_province_description','dominant_section_description','Season', 'State Region'])

one_hot_samples = pd.get_dummies(
    one_hot,
    columns=categorical_columns,
    drop_first=False
).astype(int)

encoded_samples = pd.concat([samples,one_hot_samples],axis=1)

In [8]:
### Separate Dates for modeling and case study

- **01/01/2018 - 12/31/2024** Dates to train the models
- **01/01/2025 - 01/23/2025** Dates for evaluating model performance on unseen data

In [9]:
encoded_samples['Date'] = pd.to_datetime(encoded_samples['Date'])

# first day to analyze in weather dataset
FIRST_TRAIN_DATE = pd.to_datetime('2018-01-01').date()
FIRST_TEST_DATE = pd.to_datetime('2023-12-01').date()

# last day to analyze in weather dataset
LAST_TRAIN_DATE = pd.to_datetime('2023-01-31').date()
LAST_TEST_DATE = pd.to_datetime('2025-01-22').date()

# Boolean mask for dates within the range
train_mask = (encoded_samples['Date'].dt.date >= FIRST_TRAIN_DATE) & (encoded_samples['Date'].dt.date <= LAST_TRAIN_DATE)
test_mask = (encoded_samples['Date'].dt.date >= FIRST_TEST_DATE) & (encoded_samples['Date'].dt.date <= LAST_TEST_DATE)

# Keep only rows within the range
train_model_samples = encoded_samples.loc[train_mask].copy()
test_model_samples = encoded_samples.loc[test_mask].copy()

In [10]:
display_values(train_model_samples['Target_Ignition'],
    train_model_samples['Target_Spread'],
    train_model_samples['Target_Damage'],
              console)

  Ignition Value      Spread Value       Damage Value   
      Counts             Counts             Counts      
┏━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━┳━━━━━━━━┓
┃ Index ┃  Value ┃ ┃ Index ┃  Value ┃ ┃ Index ┃  Value ┃
┡━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━╇━━━━━━━━┩
│ 0     │ 411028 │ │ 0     │ 436433 │ │ 0     │ 438088 │
│ 1     │  27223 │ │ 1     │   1818 │ │ 1     │    163 │
└───────┴────────┘ └───────┴────────┘ └───────┴────────┘

In [11]:
display_values(test_model_samples['Target_Ignition'],
    test_model_samples['Target_Spread'],
    test_model_samples['Target_Damage'],
              console)

 Ignition Value     Spread Value      Damage Value   
     Counts            Counts            Counts      
┏━━━━━━━┳━━━━━━━┓ ┏━━━━━━━┳━━━━━━━┓ ┏━━━━━━━┳━━━━━━━┓
┃ Index ┃ Value ┃ ┃ Index ┃ Value ┃ ┃ Index ┃ Value ┃
┡━━━━━━━╇━━━━━━━┩ ┡━━━━━━━╇━━━━━━━┩ ┡━━━━━━━╇━━━━━━━┩
│ 0     │ 90158 │ │ 0     │ 98349 │ │ 0     │ 98841 │
│ 1     │  8726 │ │ 1     │   535 │ │ 1     │    43 │
└───────┴───────┘ └───────┴───────┘ └───────┴───────┘

### (OPTIONAL) Subset classes to save processor
- Downsize the majority class in all three sets

In [12]:
train_model_samples['Date'] = pd.to_datetime(train_model_samples['Date'])
test_model_samples['Date'] = pd.to_datetime(test_model_samples['Date'])

In [13]:
display_values(train_model_samples['Target_Ignition'],
    train_model_samples['Target_Spread'],
    train_model_samples['Target_Damage'],
              console)

  Ignition Value      Spread Value       Damage Value   
      Counts             Counts             Counts      
┏━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━┳━━━━━━━━┓
┃ Index ┃  Value ┃ ┃ Index ┃  Value ┃ ┃ Index ┃  Value ┃
┡━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━╇━━━━━━━━┩
│ 0     │ 411028 │ │ 0     │ 436433 │ │ 0     │ 438088 │
│ 1     │  27223 │ │ 1     │   1818 │ │ 1     │    163 │
└───────┴────────┘ └───────┴────────┘ └───────┴────────┘

In [14]:
display_values(test_model_samples['Target_Ignition'],
    test_model_samples['Target_Spread'],
    test_model_samples['Target_Damage'],
              console)

 Ignition Value     Spread Value      Damage Value   
     Counts            Counts            Counts      
┏━━━━━━━┳━━━━━━━┓ ┏━━━━━━━┳━━━━━━━┓ ┏━━━━━━━┳━━━━━━━┓
┃ Index ┃ Value ┃ ┃ Index ┃ Value ┃ ┃ Index ┃ Value ┃
┡━━━━━━━╇━━━━━━━┩ ┡━━━━━━━╇━━━━━━━┩ ┡━━━━━━━╇━━━━━━━┩
│ 0     │ 90158 │ │ 0     │ 98349 │ │ 0     │ 98841 │
│ 1     │  8726 │ │ 1     │   535 │ │ 1     │    43 │
└───────┴───────┘ └───────┴───────┘ └───────┴───────┘

## Split Data

In [15]:
text_columns = ['Sample_ID', 'Date', 'grid_id',
       'geometry','area_in_cali','maximum_x', 'minimum_y',
       'maximum_y', 'minimum_x','centroid_northing','centroid_easting','Target_Damage',
                'Target_Ignition', 'Target_Spread','Year','acres','total_fire_damage',]
fire = ['fire_count','fire_count 3 Day Sum','fire_count 7 Day Sum','fire_count 30 Day Sum']
spatial = ['avg_dist_to_all_reservoirs_same_day','dist_to_reservoir_same_day',
           'dist_to_fires_same_day','avg_dist_to_fires_same_day',
           'days_since_last_fire']
drop = text_columns

## Modeling Data
train_y_ignition = train_model_samples['Target_Ignition']
train_y_spread = train_model_samples['Target_Spread']
train_y_damage = train_model_samples['Target_Damage']

test_y_ignition = test_model_samples['Target_Ignition']
test_y_spread = test_model_samples['Target_Spread']
test_y_damage = test_model_samples['Target_Damage']

train_X = train_model_samples.drop(columns=drop)
test_X = test_model_samples.drop(columns=drop)

train_details = train_model_samples[drop]
test_details  = test_model_samples[drop]


## Export Data

In [16]:
train_X.to_csv('../data/processed/train_X.csv', index=False)
train_details.to_csv('../data/processed/train_details.csv', index=False)

test_X.to_csv('../data/processed/test_X.csv', index=False)
test_details.to_csv('../data/processed/test_details.csv', index=False)

train_y_ignition.to_csv('../data/processed/train_y_ignition.csv', index=False)
train_y_spread.to_csv('../data/processed/train_y_spread.csv', index=False)
train_y_damage.to_csv('../data/processed/train_y_damage.csv', index=False)

test_y_ignition.to_csv('../data/processed/test_y_ignition.csv', index=False)
test_y_spread.to_csv('../data/processed/test_y_spread.csv', index=False)
test_y_damage.to_csv('../data/processed/test_y_damage.csv', index=False)

print("All datasets saved successfully to ../data/processed/")

All datasets saved successfully to ../data/processed/
